# 🔍 Multi-PDF Semantic RAG System
**Google Colab Version**

Upload up to **1,600 PDFs** (up to 10,000 MB total), index them into a ChromaDB vector store, then ask questions answered from the documents using Groq LLM.

In [ ]:
# Install all required dependencies
!pip install -q \
    langchain langchain-community langchain-text-splitters langchain-core \
    langchain-classic langchain-groq langchain-chroma langchain-huggingface \
    "chromadb==1.5.9" \
    sentence-transformers \
    pypdf \
    torch --index-url https://download.pytorch.org/whl/cpu \
    ipywidgets

## ⚙️ Step 1 — Configuration
Set your Groq API key below. Get one free at https://console.groq.com

In [ ]:
import os
import getpass
import uuid

# ── Groq API Key ──────────────────────────────────────────────────────────────
# Enter your Groq API key when prompted (hidden input)
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Enter your Groq API key: ")
print("✅ Groq API key set.")

# ── Session ID ────────────────────────────────────────────────────────────────
# A unique session ID isolates your vector database from other sessions.
# Re-run this cell only if you want a completely fresh session.
SESSION_ID = str(uuid.uuid4())[:8]
print(f"📂 Session ID: {SESSION_ID}")

# ── File & Size Limits ────────────────────────────────────────────────────────
MAX_FILES      = 1600        # Maximum number of PDF files per session
MAX_TOTAL_MB   = 10000       # Maximum total upload size in MB

# ── Chunking Parameters ───────────────────────────────────────────────────────
CHUNK_SIZE     = 1000        # Characters per text chunk
CHUNK_OVERLAP  = 200         # Overlap between consecutive chunks
CHUNK_BATCH_SIZE = 100       # Number of chunks sent to ChromaDB per batch

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVAL_K    = 6           # Number of top chunks to retrieve per query

# ── Embedding Model ───────────────────────────────────────────────────────────
EMBEDDING_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_NAMESPACE = "all-MiniLM-L6-v2"

# ── LLM ───────────────────────────────────────────────────────────────────────
GROQ_MODEL     = "llama-3.1-8b-instant"

# ── Storage Paths ─────────────────────────────────────────────────────────────
VECTOR_DB_ROOT = "./chroma_sessions"    # Root directory for all ChromaDB sessions
EMBED_CACHE_DIR = "./embedding_cache"   # Local cache for computed embeddings
LOG_FILE       = "./rag.log"            # Application log file
EXPERIMENT_LOG = "./experiment_runs.jsonl"  # JSONL log of indexing runs
LLM_CACHE      = "./langchain_cache.db" # SQLite cache for LLM responses

# ── Collection ────────────────────────────────────────────────────────────────
COLLECTION_NAME = "rag_collection"      # ChromaDB collection name

print("\n📋 Configuration Summary:")
print(f"  Embedding model  : {EMBEDDING_MODEL}")
print(f"  LLM model        : {GROQ_MODEL}")
print(f"  Chunk size       : {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP} chars")
print(f"  Retrieval top-k  : {RETRIEVAL_K}")
print(f"  Vector DB root   : {VECTOR_DB_ROOT}")

## 📦 Step 2 — Load Core Modules

In [ ]:
# =============================================================================
# All core modules inlined — no local .py imports needed.
# Sections: Metrics | Logging | Chunking | Embeddings | LLM |
#           VectorDB | Ingestion | Retrieval
# =============================================================================

from __future__ import annotations

import gc
import hashlib
import json
import logging
import os
import shutil
import tempfile
import threading
import time
import traceback
from contextlib import contextmanager
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import List


# =============================================================================
# === SECTION: METRICS ========================================================
# =============================================================================

@dataclass
class IndexingMetrics:
    """Tracks performance and outcome statistics for an indexing run."""
    started_at: float             = field(default_factory=time.time)
    pdf_count: int                = 0
    dataset_size_mb: float        = 0.0
    chunk_size: int               = 0
    chunk_overlap: int            = 0
    load_time: float              = 0.0
    chunk_time: float             = 0.0
    embedding_time: float         = 0.0
    vector_insertion_time: float  = 0.0
    retrieval_latency: float      = 0.0
    chroma_init_time: float       = 0.0
    total_indexing_time: float    = 0.0
    total_chunks: int             = 0
    processed_files: int          = 0
    skipped_duplicates: int       = 0
    empty_documents: int          = 0
    failed_files: int             = 0
    db_size_mb: float             = 0.0
    errors: list                  = field(default_factory=list)

    def as_dict(self):
        return {
            "pdf_count":             self.pdf_count,
            "dataset_size_mb":       round(self.dataset_size_mb, 2),
            "chunk_size":            self.chunk_size,
            "chunk_overlap":         self.chunk_overlap,
            "load_time":             round(self.load_time, 2),
            "chunk_time":            round(self.chunk_time, 2),
            "embedding_time":        round(self.embedding_time, 2),
            "vector_insertion_time": round(self.vector_insertion_time, 2),
            "retrieval_latency":     round(self.retrieval_latency, 4),
            "chroma_init_time":      round(self.chroma_init_time, 2),
            "total_indexing_time":   round(self.total_indexing_time, 2),
            "total_chunks":          self.total_chunks,
            "processed_files":       self.processed_files,
            "skipped_duplicates":    self.skipped_duplicates,
            "empty_documents":       self.empty_documents,
            "failed_files":          self.failed_files,
            "db_size_mb":            round(self.db_size_mb, 2),
            "errors":                self.errors,
        }


@dataclass
class QueryMetrics:
    """Tracks latency statistics for a single retrieval query."""
    chroma_retrieval_time: float = 0.0
    retrieval_time: float        = 0.0


@dataclass
class FileIngestionResult:
    """Result of ingesting a single PDF file."""
    file_name: str
    status: str          # 'indexed' | 'duplicate' | 'empty' | 'failed'
    chunk_count: int     = 0
    error: str | None    = None


@dataclass
class IngestionSummary:
    """Aggregate summary of an ingestion batch."""
    total_chunks: int                       = 0
    file_results: list[FileIngestionResult] = field(default_factory=list)


# =============================================================================
# === SECTION: LOGGING ========================================================
# =============================================================================

def configure_logging(log_file: str) -> None:
    """Set up file-based logging."""
    log_path = Path(log_file)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,
    )


def append_jsonl_record(path: str, payload: dict) -> None:
    """Append a single JSON record (newline-delimited) to a JSONL file."""
    record_path = Path(path)
    record_path.parent.mkdir(parents=True, exist_ok=True)
    with record_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=True) + "\n")


# =============================================================================
# === SECTION: CHUNKING =======================================================
# =============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_text_splitter() -> RecursiveCharacterTextSplitter:
    """Build the recursive character text splitter using global config values."""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " "],
        keep_separator=True,
    )


# =============================================================================
# === SECTION: EMBEDDINGS =====================================================
# =============================================================================

from langchain_classic.embeddings.cache import CacheBackedEmbeddings
from langchain_classic.storage.file_system import LocalFileStore
from langchain_huggingface import HuggingFaceEmbeddings

@lru_cache(maxsize=1)
def load_embeddings():
    """
    Load and cache the HuggingFace embedding model with a local byte-store cache.
    Uses lru_cache so the model is only loaded once per runtime session.
    """
    base_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    store = LocalFileStore(EMBED_CACHE_DIR)
    return CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings=base_embeddings,
        document_embedding_cache=store,
        namespace=EMBEDDING_NAMESPACE,
        key_encoder="sha256",
    )


# =============================================================================
# === SECTION: LLM ============================================================
# =============================================================================

from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
from langchain_groq import ChatGroq

@lru_cache(maxsize=1)
def load_llm():
    """
    Load and cache the Groq LLM with SQLite response caching.
    Uses lru_cache so the LLM client is only created once per runtime session.
    """
    set_llm_cache(SQLiteCache(database_path=LLM_CACHE))
    api_key = os.environ.get("GROQ_API_KEY") or "placeholder_key"
    return ChatGroq(api_key=api_key, model=GROQ_MODEL)


# =============================================================================
# === SECTION: VECTOR DATABASE ================================================
# =============================================================================

import chromadb
from chromadb.config import Settings
from langchain_chroma import Chroma

# Per-session threading locks so concurrent Colab cells don't corrupt ChromaDB
_session_locks: dict = {}
_locks_guard = threading.Lock()


def _get_session_lock(session_id: str) -> threading.Lock:
    """Return (and lazily create) the threading lock for a session."""
    with _locks_guard:
        if session_id not in _session_locks:
            _session_locks[session_id] = threading.Lock()
        return _session_locks[session_id]


def _session_dir(session_id: str) -> Path:
    """Return the ChromaDB directory path for a given session."""
    return Path(VECTOR_DB_ROOT) / session_id


def vector_db_exists(session_id: str) -> bool:
    """Return True if the session's vector database directory exists and is non-empty."""
    path = _session_dir(session_id)
    return path.exists() and any(path.iterdir())


def get_db_size_mb(session_id: str) -> float:
    """Return the total disk size of the session's ChromaDB in megabytes."""
    path = _session_dir(session_id)
    if not path.exists():
        return 0.0
    total = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total / (1024 * 1024)


def load_manifest(session_id: str) -> dict:
    """Load the session manifest (tracks which files have been indexed)."""
    manifest_path = _session_dir(session_id) / "manifest.json"
    if not manifest_path.exists():
        return {"indexed_files": {}}
    with open(manifest_path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_manifest(session_id: str, manifest: dict) -> None:
    """Persist the session manifest to disk."""
    path = _session_dir(session_id)
    path.mkdir(parents=True, exist_ok=True)
    with open(path / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f)


@contextmanager
def vector_db_session(session_id: str, embeddings):
    """
    Context manager that opens (or creates) a ChromaDB session.
    Acquires the per-session lock for thread safety, yields the Chroma store,
    then clears system cache and releases the lock.
    """
    lock = _get_session_lock(session_id)
    lock.acquire()
    path = _session_dir(session_id)
    path.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(
        path=str(path),
        settings=Settings(anonymized_telemetry=False, allow_reset=True),
    )
    store = Chroma(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )
    try:
        yield store
    finally:
        try:
            client.clear_system_cache()
        except Exception:
            pass
        finally:
            lock.release()


def clear_session_db(session_id: str) -> None:
    """Completely delete the vector database for a session (irreversible)."""
    lock = _get_session_lock(session_id)
    with lock:
        path = _session_dir(session_id)
        if path.exists():
            shutil.rmtree(path)


# =============================================================================
# === SECTION: INGESTION ======================================================
# =============================================================================

from langchain_community.document_loaders import PyPDFLoader


class ColabUploadedFile:
    """
    Thin wrapper around bytes returned by google.colab.files.upload().
    Mimics the interface expected by the ingestion pipeline:
      .name        — original filename
      .size        — byte length
      .getbuffer() — raw bytes
    """
    def __init__(self, name: str, data_bytes: bytes) -> None:
        self.name  = name
        self._data = data_bytes
        self.size  = len(data_bytes)

    def getbuffer(self) -> bytes:
        return self._data


def fingerprint_uploaded_file(uploaded_file) -> str:
    """Return a SHA-256 hex digest of the file's content (used for deduplication)."""
    return hashlib.sha256(uploaded_file.getbuffer()).hexdigest()


def _save_uploaded_file(uploaded_file, session_id: str) -> Path:
    """
    Write the in-memory upload to a temporary file on disk so PyPDFLoader
    can read it by path.
    """
    suffix   = Path(uploaded_file.name).suffix or ".pdf"
    temp_dir = Path(tempfile.gettempdir()) / "colab_rag_uploads" / session_id
    temp_dir.mkdir(parents=True, exist_ok=True)
    temp_path = temp_dir / f"{hashlib.md5(uploaded_file.name.encode('utf-8')).hexdigest()}{suffix}"
    with temp_path.open("wb") as handle:
        handle.write(uploaded_file.getbuffer())
    return temp_path


def _add_documents_in_batches(vector_db, chunks: list, batch_size: int):
    """
    Add document chunks to ChromaDB in fixed-size batches to avoid
    memory spikes with very large PDFs.
    Returns (total_embedding_time, total_insertion_time) in seconds.
    """
    embedding_time  = 0.0
    insertion_time  = 0.0
    for start in range(0, len(chunks), batch_size):
        batch       = chunks[start : start + batch_size]
        batch_start = time.time()
        vector_db.add_documents(batch)
        elapsed          = time.time() - batch_start
        embedding_time  += elapsed
        insertion_time  += elapsed
    return embedding_time, insertion_time


def ingest_uploaded_files(uploaded_files, session_id: str, vector_db, splitter):
    """
    Main ingestion pipeline.

    For each uploaded file:
      1. Skip if already indexed (deduplication via SHA-256 fingerprint).
      2. Save to a temp path so PyPDFLoader can load it.
      3. Split into chunks, enrich metadata, add to ChromaDB in batches.
      4. Record result in the session manifest.

    Returns (IndexingMetrics, IngestionSummary).
    """
    metrics = IndexingMetrics(
        pdf_count=len(uploaded_files),
        dataset_size_mb=sum(f.size for f in uploaded_files) / (1024 * 1024),
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    summary      = IngestionSummary()
    manifest     = load_manifest(session_id)
    indexed_files = manifest.setdefault("indexed_files", {})

    for uploaded_file in uploaded_files:
        fingerprint = fingerprint_uploaded_file(uploaded_file)

        # ── Deduplication check ───────────────────────────────────────────────
        if fingerprint in indexed_files:
            metrics.skipped_duplicates += 1
            summary.file_results.append(
                FileIngestionResult(uploaded_file.name, "duplicate")
            )
            continue

        temp_path = None
        try:
            # ── Load PDF ──────────────────────────────────────────────────────
            load_start = time.time()
            temp_path  = _save_uploaded_file(uploaded_file, session_id)
            docs       = PyPDFLoader(str(temp_path)).load()
            metrics.load_time += time.time() - load_start

            if not docs:
                metrics.empty_documents += 1
                summary.file_results.append(
                    FileIngestionResult(uploaded_file.name, "empty")
                )
                continue

            # Enrich page-level metadata
            for doc in docs:
                doc.metadata["source_pdf"]       = uploaded_file.name
                doc.metadata["session_id"]       = session_id
                doc.metadata["file_fingerprint"] = fingerprint

            # ── Chunk ─────────────────────────────────────────────────────────
            chunk_start = time.time()
            chunks      = splitter.split_documents(docs)
            if not chunks:
                metrics.empty_documents += 1
                summary.file_results.append(
                    FileIngestionResult(uploaded_file.name, "empty")
                )
                continue

            # Enrich chunk-level metadata
            total_chunks = len(chunks)
            for idx, chunk in enumerate(chunks):
                chunk.metadata.update({
                    "filename":         uploaded_file.name,
                    "session_id":       session_id,
                    "file_fingerprint": fingerprint,
                    "chunk_id":         f"{fingerprint}_{idx}",
                    "chunk_index":      idx,
                    "total_chunks":     total_chunks,
                    "page":             chunk.metadata.get("page", -1),
                    "char_count":       len(chunk.page_content),
                })
            metrics.chunk_time += time.time() - chunk_start

            # ── Embed & Insert ────────────────────────────────────────────────
            embed_time, insert_time = _add_documents_in_batches(
                vector_db, chunks, CHUNK_BATCH_SIZE
            )
            metrics.embedding_time        += embed_time
            metrics.vector_insertion_time += insert_time
            metrics.total_chunks          += len(chunks)
            metrics.processed_files       += 1

            # ── Update manifest ───────────────────────────────────────────────
            indexed_files[fingerprint] = {
                "file_name":   uploaded_file.name,
                "size_bytes":  uploaded_file.size,
                "chunk_count": len(chunks),
                "indexed_at":  time.time(),
            }
            save_manifest(session_id, manifest)
            summary.total_chunks += len(chunks)
            summary.file_results.append(
                FileIngestionResult(
                    file_name=uploaded_file.name,
                    status="indexed",
                    chunk_count=len(chunks),
                )
            )

        except Exception as exc:
            metrics.failed_files += 1
            metrics.errors.append(f"{uploaded_file.name}: {exc}")
            traceback.print_exc()
            summary.file_results.append(
                FileIngestionResult(
                    file_name=uploaded_file.name,
                    status="failed",
                    error=str(exc),
                )
            )
        finally:
            if temp_path and temp_path.exists():
                temp_path.unlink(missing_ok=True)
            gc.collect()

    metrics.total_indexing_time = time.time() - metrics.started_at
    metrics.db_size_mb          = get_db_size_mb(session_id)
    manifest.setdefault("runs", []).append(
        {"timestamp": time.time(), "metrics": metrics.as_dict()}
    )
    save_manifest(session_id, manifest)
    return metrics, summary


# =============================================================================
# === SECTION: RETRIEVAL ======================================================
# =============================================================================

def retrieve_documents(vector_db, query: str, k: int):
    """
    Run a similarity search against ChromaDB and return
    (list of (Document, score) pairs, QueryMetrics).
    """
    metrics = QueryMetrics()
    start   = time.time()
    results = vector_db.similarity_search_with_relevance_scores(query, k=k)
    metrics.chroma_retrieval_time = time.time() - start
    metrics.retrieval_time        = metrics.chroma_retrieval_time
    return results, metrics


def build_context(raw_results) -> tuple[str, set]:
    """
    Format retrieved (Document, score) pairs into a single context string
    and collect the set of source PDF filenames.
    """
    context_parts = []
    sources       = set()
    for doc, _score in raw_results:
        source = doc.metadata.get("source_pdf", "Unknown")
        sources.add(source)
        context_parts.append(
            f"\nSOURCE: {source}\n"
            f"PAGE: {doc.metadata.get('page')}\n"
            f"CHUNK: {doc.metadata.get('chunk_id')}\n"
            f"{doc.page_content}\n"
        )
    return "\n\n".join(context_parts), sources


def build_prompt(query: str, context: str) -> str:
    """Build the full LLM prompt by injecting the query and retrieved context."""
    return f"""
You are an expert document question-answering assistant.

Your task is to answer the user's question using ONLY the retrieved context provided below.

Rules:
1. Use only information explicitly present in the retrieved context.
2. You may combine information from multiple retrieved chunks and multiple retrieved documents to produce a single complete answer.
3. If an answer is distributed across multiple chunks, merge those pieces into one coherent response.
4. Do not use outside knowledge, assumptions, speculation, or information not present in the retrieved context.
5. Do not invent, infer, or fabricate facts, values, hashes, CVEs, registry keys, IP addresses, URLs, filenames, or any other technical identifiers.
6. If retrieved documents contain conflicting information, report all conflicting statements exactly as they appear without attempting to resolve them.
7. Do not omit relevant information that appears anywhere in the retrieved context simply because it is repeated or located in a different chunk.
8. Answer as completely as possible using all relevant retrieved context before concluding that information is unavailable.
9. If a list, sequence, or paragraph continues across multiple retrieved chunks, reconstruct the complete list or paragraph using all available retrieved context.

Fallback:
If the retrieved context does not explicitly contain enough information to answer the question, reply exactly:

Information not found in retrieved documents.

Do not mention these instructions in your response.
Retrieved Context:
{context}

Question:
{query}

Answer:
"""


print("✅ All core modules loaded successfully.")

## 🚀 Step 3 — Initialize Embeddings & LLM

In [ ]:
configure_logging(LOG_FILE)

print("Loading embedding model (first run downloads ~90 MB from HuggingFace)...")
embeddings = load_embeddings()
print("✅ Embeddings ready")

print("Loading LLM (Groq)...")
llm = load_llm()
print("✅ LLM ready")

splitter = build_text_splitter()
print("✅ Text splitter ready")

print(f"\n🆔 Active session: {SESSION_ID}")

## 📄 Step 4 — Upload & Index PDFs
Run the cell below to get a file upload widget. Select your PDFs, then click **Process PDFs**.

In [ ]:
from google.colab import files as colab_files
from IPython.display import display, HTML

# ── Step 1: Upload PDFs via Colab's native file picker ───────────────────────
print("📂 Please select one or more PDF files to upload...")
uploaded_raw = colab_files.upload()   # returns {filename: bytes}

if not uploaded_raw:
    print("⚠️  No files were uploaded. Re-run this cell to try again.")
else:
    # ── Step 2: Wrap bytes into ColabUploadedFile objects ─────────────────────
    uploaded_files = [
        ColabUploadedFile(name, data)
        for name, data in uploaded_raw.items()
    ]

    # ── Step 3: Validate limits ───────────────────────────────────────────────
    total_count  = len(uploaded_files)
    total_size_mb = sum(f.size for f in uploaded_files) / (1024 * 1024)

    print(f"\n📋 Upload Summary:")
    print(f"  Files uploaded : {total_count}")
    print(f"  Total size     : {total_size_mb:.2f} MB")

    if total_count > MAX_FILES:
        print(f"❌ Too many files: {total_count} uploaded, limit is {MAX_FILES}.")
    elif total_size_mb > MAX_TOTAL_MB:
        print(f"❌ Total size too large: {total_size_mb:.2f} MB, limit is {MAX_TOTAL_MB} MB.")
    else:
        print("\n⏳ Processing PDFs — this may take a while for large batches...\n")

        # ── Step 4: Ingest inside the vector_db_session context manager ───────
        with vector_db_session(SESSION_ID, embeddings) as store:
            metrics, summary = ingest_uploaded_files(
                uploaded_files, SESSION_ID, store, splitter
            )

        # ── Step 5: Display results table ─────────────────────────────────────
        print("\n" + "=" * 60)
        print("  📊 INDEXING RESULTS")
        print("=" * 60)

        rows = ""
        status_emoji = {"indexed": "✅", "duplicate": "⏭️", "empty": "📭", "failed": "❌"}
        for result in summary.file_results:
            emoji = status_emoji.get(result.status, "❓")
            chunk_info = f"{result.chunk_count} chunks" if result.status == "indexed" else result.status
            error_info = f" — {result.error}" if result.error else ""
            rows += (
                f"<tr>"
                f"<td>{emoji}</td>"
                f"<td style='padding:2px 8px'>{result.file_name}</td>"
                f"<td style='padding:2px 8px'>{chunk_info}{error_info}</td>"
                f"</tr>"
            )

        display(HTML(
            "<table style='border-collapse:collapse;font-size:13px'>"
            "<tr><th>Status</th><th style='padding:2px 8px'>File</th><th style='padding:2px 8px'>Details</th></tr>"
            + rows +
            "</table>"
        ))

        m = metrics.as_dict()
        print(f"\n📈 Metrics:")
        print(f"  Processed     : {m['processed_files']} file(s)")
        print(f"  Skipped (dup) : {m['skipped_duplicates']} file(s)")
        print(f"  Empty         : {m['empty_documents']} file(s)")
        print(f"  Failed        : {m['failed_files']} file(s)")
        print(f"  Total chunks  : {m['total_chunks']}")
        print(f"  DB size       : {m['db_size_mb']:.2f} MB")
        print(f"  Total time    : {m['total_indexing_time']:.2f}s")
        print(f"    • Load      : {m['load_time']:.2f}s")
        print(f"    • Chunk     : {m['chunk_time']:.2f}s")
        print(f"    • Embed+Insert: {m['embedding_time']:.2f}s")

        if m["errors"]:
            print("\n⚠️  Errors:")
            for err in m["errors"]:
                print(f"  • {err}")

        print("\n✅ Indexing complete. Proceed to Step 5 to ask questions.")

## 💬 Step 5 — Ask Questions

In [ ]:
# Change this question and re-run the cell to ask different questions
QUERY = "What are the main tactics used by APT groups?"   # <-- edit this

from IPython.display import Markdown, display

if not vector_db_exists(SESSION_ID):
    print("❌ No indexed PDFs found. Run Step 4 first.")
else:
    print(f"🔍 Querying: {QUERY!r}")
    print("⏳ Retrieving relevant chunks...")

    with vector_db_session(SESSION_ID, embeddings) as store:
        raw_results, query_metrics = retrieve_documents(store, QUERY, RETRIEVAL_K)

    context, sources = build_context(raw_results)
    prompt           = build_prompt(QUERY, context)

    print("🤖 Generating answer with Groq LLM...")
    response = llm.invoke(prompt)
    answer   = response.content

    display(Markdown(f"## 🤖 Answer\n\n{answer}"))
    display(Markdown(
        f"### 📚 Source PDFs\n" +
        "\n".join(f"- {s}" for s in sorted(sources))
    ))

    print(f"\n📊 Retrieval Metrics:")
    print(f"  Chroma retrieval time : {query_metrics.chroma_retrieval_time:.4f}s")
    print(f"  Total retrieval time  : {query_metrics.retrieval_time:.4f}s")

    print("\n📄 Retrieved Chunks:")
    for i, (doc, score) in enumerate(raw_results, 1):
        print(f"\n--- Chunk {i} ---")
        print(f"  Source : {doc.metadata.get('source_pdf')}")
        print(f"  Page   : {doc.metadata.get('page')}")
        print(f"  Score  : {score:.4f}")
        print(f"  Text   : {doc.page_content[:300]}...")

## 🛠️ Utilities

In [ ]:
# Run this cell to wipe the vector database and start fresh
clear_session_db(SESSION_ID)
print("✅ Vector database cleared.")